In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import os

import pandas as pd 

from time import sleep

from selenium import webdriver

from selenium.webdriver.common.by import By

from pandas import ExcelWriter

import datetime

import os

from selenium.webdriver.chrome.service import Service as ChromeService



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CO SFCO' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {'CO SFCO 1': 'https://www.datos.gov.co/Hacienda-y-Cr-dito-P-blico/Entidades-vigiladas-por-la-Superfinanciera/sr9n-792w/data_preview', }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



col_id =['tipo_entidad','cod_entidad','razon_social','direccion','ciudad','numeroidentificacion', 'nombrepublicocargo','representante_legal','emailprincipal','uripaginaweb' ]

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]





# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):



  print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_")

  driver.get(regdict[reg])

  sleep(2)

  driver.find_element(By.XPATH, '//*[@id="view-switcher-nav-bar"]/div/div/forge-button').click()

  sleep(3)

  driver.find_element(By.XPATH, '/html/body/forge-dialog/div/forge-scaffold/div[3]/forge-toolbar/forge-button[2]/button').click()

  file =   check_dowload_files(tempfolder, 'CSV')

  filePath = os.path.join(tempfolder, file)

  df = pd.read_csv(filePath)



  print(f"[INFO] : -- DataFrame '{file}' | containe = {df.shape}")

  for index, row in df.iterrows():



    sqldict['Name'].append(row['RAZON_SOCIAL'])

    sqldict['InternalID_1'].append(row['COD_ENTIDAD'])

    sqldict['InternalID_1_type'].append('ENTITY CODE')

    sqldict['InternalID_2'].append(row['NUMEROIDENTIFICACION'])

    sqldict['InternalID_2_type'].append('NUMERO IDENTIFICAION')

    sqldict['Email'].append(row['EMAILPRINCIPAL'])

    Website = '' if row['URIPAGINAWEB'] == 'Pendiente' or row['URIPAGINAWEB'] =='N/A.' else row['URIPAGINAWEB']

    sqldict['Website'].append(Website)

    sqldict['Address_1'].append(row['DIRECCION'])

    sqldict['City'].append(row['CIUDAD'])

    # sqldict['Typology'].append(row['TIPO_ENTIDAD']) 

    # sqldict['EntryType'].append(row['REPRESENTANTE_LEGAL'])

    sqldict["RegulationType"].append("supervised")

    sqldict["ListName"].append("Entidades supervisadas")

    sqldict['ListProcessDate'].append(processdate)

    sqldict['RegCtry'].append(reg.split(' ')[0])

    sqldict['RegCode'].append(reg.split(' ')[1])

    sqldict['ListCode'].append(reg.split(' ')[-1])



  sqldict = bourange_same_length_array(sqldict)



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


    